# 5 — The autodiff cheat sheet

Reverse-mode AD for this language, end to end: the IR, the adjoint of every
instruction (with a one-line derivation each), numerical validation, and the
chain-rule machinery that assembles them.

**The one derivation trick.** Every layout op is a *linear* map L on the
value space. Define the inner product ⟨a, b⟩ = Σ over all coordinates of
a·b. Then the cotangent (gradient) pulls back through the *adjoint*:

    ⟨L x, y⟩ = ⟨x, L† y⟩      ⇒      x̄ = L†(ȳ)

Derive any adjoint by writing the left side as a sum and re-grouping it
around x. Nonlinear ops (pointwise markers) contribute their per-element
partial times the incoming cotangent — the chain rule. Two more rules
complete the system: **fan-out accumulates** (a value used twice gets its
cotangents added) and **gradient-free carriers** (bool/int values — masks,
iota, comparisons — carry no cotangent).

In [1]:
import numpy as np
from nbhelp import show  # also puts tensorlib on sys.path
from tensorlib import Tensor
from tensorlib.autodiff import grad, numeric_grad
from tensorlib.ir import Instr, Program, run

## The IR in sixty seconds

A program is a straight line of (var, op, operands, params). No branching:
the tape IS the program, read backwards.

In [2]:
def instr(var, op, operands=(), **params):
    return Instr(var, op, tuple(operands), params)


def T(arr, names):
    return Tensor.from_numpy(np.asarray(arr, dtype=np.float64), names)


def validate(instrs, inputs, wrt, target=None):
    prog = Program(tuple(instrs))
    target = target or prog.instrs[-1].var
    joint, grads = grad(prog, target, dict(inputs))
    env = run(joint, inputs)
    got = env[grads[wrt]].to_numpy(order=inputs[wrt].names)
    want = numeric_grad(prog, target, wrt, inputs)
    ok = np.allclose(got, want, rtol=1e-4, atol=1e-6)
    print(("OK " if ok else "FAIL") + f"  d{target}/d{wrt}   max|Δ| = {np.abs(got - want).max():.2e}")
    return joint, grads, env


rng = np.random.default_rng(0)

## repeat ⊣ reduce-sum — the fundamental pair

    ⟨repeat(x), y⟩ = Σᵢₙ xᵢ yᵢₙ = Σᵢ xᵢ (Σₙ yᵢₙ) = ⟨x, reduceSumₙ(y)⟩

Broadcast's adjoint is summation, and vice versa. Everything with
"normalization" in its name is built from this pair.

In [3]:
p = [
    instr("x", "input"),
    instr("r", "repeat", ["x"], name="n", extent=(0, 3)),
    instr("w", "input"),
    instr("m", "pointwise", ["r", "w"], f="mul"),
    instr("y", "reduce", ["m"], f="sum", dims=("i", "n")),
]
validate(p, {"x": T(rng.standard_normal(4), ("i",)), "w": T(rng.standard_normal((4, 3)), ("i", "n"))}, "x");

OK   dy/dx   max|Δ| = 1.60e-10


## slice ⊣ pad — restriction and zero-extension

    ⟨slice(x), y⟩ = Σ_{i∈S} xᵢ yᵢ = ⟨x, pad₀(y)⟩

The cotangent of a slice is the cotangent zero-extended back to the full
domain — and pad's adjoint is the slice, with cotangent arriving in the
fill region *discarded* (fill is a constant; nothing flows to it).

In [4]:
p = [
    instr("x", "input"),
    instr("s", "slice", ["x"], ranges={"i": (1, 4)}),
    instr("y", "reduce", ["s"], f="sum", dims=("i",)),
]
joint, grads, env = validate(p, {"x": T([1.0, 2, 3, 4, 5], ("i",))}, "x")
show(env[grads["x"]], "d y / d x  — ones inside the slice, zeros outside")
print(env[grads["x"]].to_numpy())

OK   dy/dx   max|Δ| = 1.03e-09
-- d y / d x  — ones inside the slice, zeros outside
Tensor[float64] on Buffer(8B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  i            0      0      5      5  
  numel=5  footprint=(0, 8)  injectivity=unknown
  Guard(1 <= i < 4)
  fill   : 0.0
[0. 1. 1. 1. 0.]


## Relabelings are (almost) their own adjoints

shift ⊣ shift-back, flip ⊣ flip, rename ⊣ inverse-rename, split ⊣ merge,
merge ⊣ split — coordinate relabelings move the cotangent by the inverse
relabeling. (The generated split-adjoint inserts a `materialize` before its
merge: merging needs real stride nesting, and a cotangent arriving from an
arbitrary chain may not have it.)

In [5]:
p = [
    instr("x", "input"),
    instr("b", "split", ["x"], name="i", parts={"io": 2, "ii": 3}),
    instr("w", "input"),
    instr("m", "pointwise", ["b", "w"], f="mul"),
    instr("mo", "materialize", ["m"], order=("io", "ii")),
    instr("g", "merge", ["mo"], parts=("io", "ii"), name="i"),
    instr("y", "reduce", ["g"], f="sum", dims=("i",)),
]
validate(p, {"x": T(rng.standard_normal(6), ("i",)), "w": T(rng.standard_normal((2, 3)), ("io", "ii"))}, "x");

OK   dy/dx   max|Δ| = 9.49e-11


## window / stencil ⊣ overlap-add — the convolution adjoint

    ⟨W x, y⟩ = Σₐₖ x_{a+k} yₐₖ = Σⱼ xⱼ (Σₖ y_{j−k,k})   ⇒   x̄ⱼ = Σₖ ȳ_{j−k,k}

Overlapping windows read each xⱼ many times; the adjoint *adds back* each
tap's cotangent, shifted. The transform unrolls per tap: select the tap,
shift by it, slice to the valid region (a stencil's out-of-guard taps die
here — fill gets no gradient), pad with zeros, accumulate.

In [6]:
p = [
    instr("x", "input"),
    instr("xs", "stencil", ["x"], name="i", k=(-1, 1), fill=0.0),
    instr("w", "input"),
    instr("wr", "repeat", ["w"], name="i", extent=(0, 5)),
    instr("m", "pointwise", ["xs", "wr"], f="mul"),
    instr("y", "reduce", ["m"], f="sum", dims=("i", "i_k")),
]
inputs = {"x": T(rng.standard_normal(5), ("i",)), "w": T(rng.standard_normal(3), ("i_k",)).shift(i_k=-1)}
joint, grads, env = validate(p, inputs, "x")
print()
print("the generated overlap-add (backward excerpt):")
for ins2 in joint.instrs[len(p) :]:
    if ins2.op in ("select", "shift", "slice", "pad"):
        print(" ", ins2)

OK   dy/dx   max|Δ| = 7.97e-11

the generated overlap-add (backward excerpt):
  %g8 = select(%g4; coords={'i_k': -1})
  %g9 = shift(%g8; deltas={'i': -1})
  %g10 = slice(%g9; ranges={'i': (0, 4)})
  %g11 = pad(%g10; fill=0.0, extents={'i': (0, 5)})
  %g12 = select(%g4; coords={'i_k': 0})
  %g13 = shift(%g12; deltas={'i': 0})
  %g14 = slice(%g13; ranges={'i': (0, 5)})
  %g15 = pad(%g14; fill=0.0, extents={'i': (0, 5)})
  %g16 = select(%g4; coords={'i_k': 1})
  %g17 = shift(%g16; deltas={'i': 1})
  %g18 = slice(%g17; ranges={'i': (1, 5)})
  %g19 = pad(%g18; fill=0.0, extents={'i': (0, 5)})


## decimate ⊣ zero-stuffing

    ⟨D x, y⟩ = Σⱼ x_{fj+p} yⱼ   ⇒   x̄ᵢ = ȳ_{(i−p)/f} when i ≡ p (mod f), else 0

The adjoint spreads the cotangent into every f-th slot. Generated as:
repeat a phase dim, mask the kept slot with an iota comparison (masks are
gradient-free by carrier), materialize in interleave order, merge.

In [7]:
p = [
    instr("x", "input"),
    instr("d", "decimate", ["x"], name="i", factor=2, phase=1),
    instr("w", "input"),
    instr("m", "pointwise", ["d", "w"], f="mul"),
    instr("y", "reduce", ["m"], f="sum", dims=("i",)),
]
inputs = {"x": T(rng.standard_normal(6), ("i",)), "w": T(rng.standard_normal(3), ("i",))}
joint, grads, env = validate(p, inputs, "x")
print("gradient:", env[grads["x"]].to_numpy(), " (zeros on the dropped phase)")

OK   dy/dx   max|Δ| = 1.23e-10
gradient: [0.         1.34587542 0.         0.7813114  0.         0.26445563]  (zeros on the dropped phase)


## diagonal ⊣ masked embedding

    ⟨diag(x), y⟩ = Σ_z x_{z,z} y_z   ⇒   x̄ᵢⱼ = [i = j] · ȳᵢ

The cotangent lands on the diagonal of a zero matrix — built from an iota
equality mask and a zero constant, then padded to the full box.

In [8]:
p = [
    instr("x", "input"),
    instr("d", "diagonal", ["x"], parts=("i", "j"), name="z"),
    instr("y", "reduce", ["d"], f="sum", dims=("z",)),
]
joint, grads, env = validate(p, {"x": T(rng.standard_normal((3, 3)), ("i", "j"))}, "x")
print(env[grads["x"]].to_numpy(order=("i", "j")))

OK   dy/dx   max|Δ| = 1.40e-10
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## The pointwise marker table

| marker | x̄ (first operand) | second operand |
|---|---|---|
| add | c | c |
| sub | c | −c |
| mul | c·B | c·A |
| div | c/B | −c·A/B² |
| exp | c·out (reuses the forward output!) | — |
| log | c/A | — |
| maximum | c·[A≥B] | c·[B>A] (ties to the first) |
| where(m,a,b) | routed by m | routed by ¬m; m itself: none (bool) |
| comparisons, iota | gradient-free (int/bool carriers) | |

## reduce and scan

| op | adjoint |
|---|---|
| reduce(sum) | repeat |
| reduce(mean) | repeat, then ÷N (N static, from the layout) |
| reduce(max) | repeat, masked by eq(A, max) — tie caveat |
| scan(sum) | reverse scan: flip ∘ scan(sum) ∘ flip |

Derivation for scan: yₜ = Σ_{s≤t} xₛ ⇒ x̄ₛ = Σ_{t≥s} ȳₜ — suffix sums.

In [9]:
p = [
    instr("x", "input"),
    instr("cs", "scan", ["x"], f="sum", dim="i"),
    instr("w", "input"),
    instr("m", "pointwise", ["cs", "w"], f="mul"),
    instr("y", "reduce", ["m"], f="sum", dims=("i",)),
]
validate(p, {"x": T(rng.standard_normal(5), ("i",)), "w": T(rng.standard_normal(5), ("i",))}, "x");

OK   dy/dx   max|Δ| = 1.07e-10


## The chain rule, mechanically

`grad` walks the program backwards. Each variable collects cotangent
*contributions* from every instruction that consumed it; fan-out means
several contributions, summed. Watch x used twice in x·x:

In [10]:
p = [
    instr("x", "input"),
    instr("sq", "pointwise", ["x", "x"], f="mul"),
    instr("y", "reduce", ["sq"], f="sum", dims=("i",)),
]
joint, grads, env = validate(p, {"x": T([1.0, -2.0, 3.0], ("i",))}, "x")
print("d/dx Σ x² =", env[grads["x"]].to_numpy(), " (= 2x: two contributions, added)")
print()
print("the full joint program:")
print(joint)

OK   dy/dx   max|Δ| = 8.39e-10
d/dx Σ x² = [ 2. -4.  6.]  (= 2x: two contributions, added)

the full joint program:
x = input()
sq = pointwise(x, x; f='mul')
y = reduce(sq; f='sum', dims=('i',))
%seed0 = const(; value=1.0, dims=())
%g1 = repeat(%seed0; name='i', extent=(0, 3))
%s2 = strip_charts(x)
%g3 = pointwise(%g1, %s2; f='mul')
%g4 = pointwise(%g1, %s2; f='mul')
%acc5 = pointwise(%g3, %g4; f='add')


## Seeds: gradients are vector-Jacobian products

A scalar target seeds with 1 automatically. A non-scalar target has a
Jacobian, and reverse mode natively computes v↦Jᵀv — so a seed is
*required*, as an extra runtime input aligned with the target (silent
ones-seeding is a footgun this transform refuses).

In [11]:
p = Program(
    (
        instr("x", "input"),
        instr("d", "pointwise", ["x", "x"], f="mul"),
    )
)
try:
    grad(p, "d", {"x": T([1.0, 2.0], ("i",))})
except ValueError as e:
    print("refused:", e)
joint, grads = grad(p, "d", {"x": T([1.0, 2.0], ("i",))}, seed="dY")
env = run(joint, {"x": T([3.0, 4.0], ("i",)), "dY": T([1.0, 0.0], ("i",))})
print("row of the Jacobian via seed [1,0]:", env[grads["x"]].to_numpy())

refused: target 'd' is not a scalar; pass seed= (the name of a runtime input aligned with the target) — reverse mode computes vector-Jacobian products
row of the Jacobian via seed [1,0]: [6. 0.]


## End to end: training with the language

Softmax cross-entropy has the famous analytic gradient softmax(S) − onehot —
the generated program reproduces it. Then: a few steps of gradient descent
on a linear model, driven entirely by IR programs.

In [12]:
s = rng.standard_normal((2, 4))
onehot = np.zeros((2, 4))
onehot[0, 1] = onehot[1, 3] = 1.0
p = [
    instr("S", "input"),
    instr("t", "input"),
    instr("mx", "reduce", ["S"], f="max", dims=("v",)),
    instr("mr", "repeat", ["mx"], name="v", extent=(0, 4)),
    instr("sh", "pointwise", ["S", "mr"], f="sub"),
    instr("e", "pointwise", ["sh"], f="exp"),
    instr("z", "reduce", ["e"], f="sum", dims=("v",)),
    instr("zr", "repeat", ["z"], name="v", extent=(0, 4)),
    instr("lz", "pointwise", ["zr"], f="log"),
    instr("lp", "pointwise", ["sh", "lz"], f="sub"),
    instr("nll", "pointwise", ["lp", "t"], f="mul"),
    instr("s1", "reduce", ["nll"], f="sum", dims=("v",)),
    instr("nl", "pointwise", ["s1"], f="neg"),
    instr("L", "reduce", ["nl"], f="sum", dims=("i",)),
]
inputs = {"S": T(s, ("i", "v")), "t": T(onehot, ("i", "v"))}
joint, grads = grad(Program(tuple(p)), "L", dict(inputs))
env = run(joint, inputs)
sm = np.exp(s - s.max(1, keepdims=True))
sm /= sm.sum(1, keepdims=True)
print("max |dL/dS − (softmax − onehot)| =", np.abs(env[grads["S"]].to_numpy(order=("i", "v")) - (sm - onehot)).max())

max |dL/dS − (softmax − onehot)| = 5.551115123125783e-17


In [13]:
xd = rng.standard_normal((8, 3))
true_w = np.array([2.0, -1.0, 0.5])
yd = xd @ true_w
p = [
    instr("X", "input"),
    instr("Wp", "input"),
    instr("Y", "input"),
    instr("X3", "repeat", ["X"], name="o", extent=(0, 1)),
    instr("W3", "repeat", ["Wp"], name="i", extent=(0, 8)),
    instr("P", "pointwise", ["X3", "W3"], f="mul"),
    instr("yh", "reduce", ["P"], f="sum", dims=("k",)),
    instr("Y3", "repeat", ["Y"], name="o", extent=(0, 1)),
    instr("r", "pointwise", ["yh", "Y3"], f="sub"),
    instr("r2", "pointwise", ["r", "r"], f="mul"),
    instr("L", "reduce", ["r2"], f="mean", dims=("i", "o")),
]
w = np.zeros(3)
prog = Program(tuple(p))
joint, grads = grad(prog, "L", {"X": T(xd, ("i", "k")), "Wp": T(np.zeros((3, 1)), ("k", "o")), "Y": T(yd, ("i",))})
for step in range(60):
    inputs = {"X": T(xd, ("i", "k")), "Wp": T(w[:, None], ("k", "o")), "Y": T(yd, ("i",))}
    env = run(joint, inputs)
    g = env[grads["Wp"]].to_numpy(order=("k", "o"))[:, 0]
    w -= 0.4 * g
    if step % 20 == 0 or step == 59:
        print(f"step {step:2d}   loss = {float(env['L'].item()):.6f}")
print("learned w =", np.round(w, 4), "   true w =", true_w)

step  0   loss = 6.637535
step 20   loss = 0.000000
step 40   loss = 0.000000
step 59   loss = 0.000000
learned w = [ 2.  -1.   0.5]    true w = [ 2.  -1.   0.5]


---
Known gaps, honestly (CONCERNS #19–21): cotangents live on the lattice
(charts are stripped — whether gradients should inherit charts is open);
decimate's adjoint needs a factor-divisible domain; only scan(sum) and
reduce(sum/mean/max/min) differentiate so far; n-ary diagonal adjoints and
pair-state scans await the marker DSL. Next layer up: REPRESENTATIONS.md —
schedules, checkpointing, and memory.